# SmolLM2-recurrent continued pretraining

Upload this notebook to Colab (Runtime > Change runtime type > GPU) and Run All. No editing required -- except the one `MODE` line in the cell below.

One-time setup, before the first run: open the key icon in the left sidebar (Secrets) and add:
- `HF_TOKEN` (required) -- a Hugging Face token with **write** access, from https://huggingface.co/settings/tokens. Used to pull/push checkpoints and the final model, so no Drive mount is needed.
- `WANDB_API_KEY` (optional) -- from https://wandb.ai/authorize. If missing, training still runs fine, just prints loss to the cell output instead of logging to wandb.

`MODE = "smoke_test"` (the default) runs ~200 steps on a couple hundred MB of data -- just enough to prove the pipeline works end-to-end (data mix, train, checkpoint push, resume, eval) and to sanity-check the loss and recurrence-depth trend before spending real GPU hours. Flip it to `MODE = "full_run"` for the actual ~7500-step / ~500M-token continued-pretraining run once the smoke test looks good.

`full_run` trains a two-phase curriculum toward beating base SmolLM2-360M, not just a bigger smoke test: phase 1 "heals" the model on plain FineWeb-Edu after the recurrence retrofit, then phase 2 diversifies into FineWeb-Edu/DCLM/Cosmopedia-v2 plus reasoning-heavy math and code data (IFM/Math-Reasoning, IFM/Code-Reasoning -- both ungated, Apache-2.0), and switches the optimizer to Muon. The mean-recurrence ceiling is also raised to 8 (from the smoke test's 4). See `mix_smollm2_corpus.py` and the `MODE` cell below for the full rationale -- this is a budget-scaled version of arXiv:2511.07384's recipe, not a full replication (the paper trains to recurrence 16-32 over 52B tokens, which is weeks/hundreds of dollars on a single GPU).

Everything is on the Hub, not Google Drive, and everything is namespaced by run name so the two modes never collide or resume from each other's state:
- Training data re-mixes into local (ephemeral) Colab disk each session -- it isn't persisted, so a fresh session re-downloads/re-mixes it. `full_run`'s two phases mix into separate subdirectories and train back-to-back within the same run.
- The **latest** full resumable checkpoint (weights + optimizer + dataloader state) for the current run is pushed to the private dataset repo `usr-wwelsh/smollm2-recurrent-checkpoints` every save, overwriting that run's previous one -- so it never accumulates, and there's always exactly one copy to resume from. This one checkpoint carries the run across both `full_run` phases too -- which phase resumes is decided by comparing its saved step against each phase's step boundary.
- Every time you reopen this notebook and Run All again (same `MODE`), it checks that repo for a checkpoint under the current run name and resumes from it instead of starting over. Expect to need this a lot -- free-tier Colab sessions time out well before a full run finishes, and `smoke_test` checkpoints deliberately save often (every 25 steps) so you can kill the runtime mid-test and confirm resume actually works.
- Only `full_run` pushes the finished weights to the public model repo `usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4-trained` -- `smoke_test` never publishes anything.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduces allocator fragmentation -- the smoke test OOM'd with 738MB reserved-but-unallocated while only 80MB was actually free

import torch

assert torch.cuda.is_available(), (
    "No GPU visible. Go to Runtime > Change runtime type and select a GPU, "
    "then Runtime > Restart session and run this cell again."
)

gpu_name = torch.cuda.get_device_name(0)
major, _minor = torch.cuda.get_device_capability(0)
no_amp = "false" if major >= 8 else "true"
print(f"GPU: {gpu_name} (compute capability {major}.{_minor}) -> no_amp={no_amp}")
print(
    "bf16 autocast enabled." if no_amp == "false" else
    "Pre-Ampere GPU (T4/P100) -- no native bf16, running in fp32 for correctness (slower per-step, but correct)."
)

# fp32 (no_amp) roughly doubles activation memory vs bf16 -- a T4's 14.6GB fills up fast, and
# mean_recurrence ramping 1->4 over the warmup window grows peak memory further as training
# progresses (not just at step 0). micro_batch_size=2 still OOM'd at update 13 (14.48/14.56GB
# used, 738MB reserved-but-unallocated -- allocator fragmentation, addressed above) once
# recurrence depth climbed partway up its ramp, so 1 is the floor for T4/no_amp.
# batch_size stays fixed at 64 either way -- a smaller micro_batch_size just means more gradient
# accumulation steps to reach it, not a smaller effective batch.
MICRO_BATCH_SIZE = 1 if no_amp == "true" else 8
print(f"micro_batch_size={MICRO_BATCH_SIZE} (gradient-accumulated up to batch_size=64)")

In [ ]:

# ============================================================================
# The one line to change: "smoke_test" to prove the pipeline works cheaply,
# "full_run" for the real continued-pretraining run. Everything below derives
# from this -- run name, step count, token budget, checkpoint frequency.
MODE = "smoke_test"
# ============================================================================

if MODE == "smoke_test":
    RUN_NAME = "smollm2-recurrent-smoketest"
    MAX_STEPS = 200
    TOKEN_BUDGET = 15_000_000  # 200 steps * batch_size=64 * max_length=1024 needs ~13.1M tokens minimum
    SAVE_INTERVAL = 25  # eval checkpoint every 25 steps, full resumable checkpoint every 50 -- frequent on purpose, so a session kill mid-test actually exercises resume
    ROWS_PER_SHARD = 2_000  # ~14.6k rows expected total -- flush every ~2M tokens instead of only at the very end, so a kill mid-mix still leaves usable shards and you see "wrote shard-*" early instead of nothing
    LOG_EVERY = 200  # default (1000) can go a couple minutes with zero output on a small budget -- looks hung even though it isn't
    SHUFFLE_BUFFER_SIZE = 0  # streaming .shuffle() yields nothing until its buffer fills -- default 10,000 means 3 sources * 10k = 30k documents downloaded before the FIRST packed row, dwarfing this test's ~14.6k total rows. 0 disables shuffling (mix ordering is a non-issue at this scale) so packing starts immediately
    MAX_MEAN_REC = 4  # single-GPU-scale reference (shells/tinyllama.sh) value -- just a pipeline check, not the real curriculum
    MAX_BACKPROP = 2
    USE_MUON = False  # keep the smoke test on plain AdamW -- it's only proving the pipeline works, not testing the optimizer
    # Single phase: DCLM stays off (its ~2.9GB single-row-group shards need ~9GB+ RAM to
    # read even one row from -- no benefit for a 200-step pipeline check), and the
    # reasoning sources stay off too since the smoke test isn't evaluating curriculum quality.
    PHASES = [
        dict(
            name="smoketest",
            end_step=MAX_STEPS,
            token_budget=TOKEN_BUDGET,
            weights=dict(fineweb_edu_weight=0.60, dclm_weight=0.0, cosmopedia_v2_weight=0.04, math_reasoning_weight=0.0, code_reasoning_weight=0.0),
        )
    ]
elif MODE == "full_run":
    RUN_NAME = "smollm2-recurrent-v1"
    MAX_STEPS = 7500
    TOKEN_BUDGET = 500_000_000
    SAVE_INTERVAL = 250
    ROWS_PER_SHARD = 20_000  # default -- unchanged from the original design
    LOG_EVERY = 1_000
    SHUFFLE_BUFFER_SIZE = 10_000  # default -- fine here, the 30k-doc prefetch is negligible against 500M tokens
    MAX_MEAN_REC = 8  # paper trains to 16-32, but that's ~weeks/hundreds of dollars on a single GPU -- 8 is the notes' budget-realistic compromise, still well above the smoke test's 4
    MAX_BACKPROP = 4  # kept at half of MAX_MEAN_REC, mirroring the smoke test's 4:2 ratio -- truncated backprop is what makes the higher recurrence ceiling affordable memory-wise
    USE_MUON = True  # train.py already supports --muon.use_muon -- the paper's recipe uses Muon, not plain AdamW

    # Two-phase curriculum (arXiv:2511.07384): phase 1 "heals" the model on
    # plain FineWeb-Edu after the recurrence retrofit, phase 2 diversifies
    # into the full web mix plus reasoning-heavy math/code data. The paper's
    # own math ingredient (nvidia/Nemotron-CC-Math-v1) sits behind a
    # corporate-only license click-through, so phase 2 uses two ungated,
    # Apache-2.0 alternatives instead -- IFM/Math-Reasoning (worked
    # chain-of-thought math) and IFM/Code-Reasoning (reasoning-annotated
    # coding problems) -- see mix_smollm2_corpus.py's docstring.
    #
    # Phase 1's token budget is sized so it runs out right around a full
    # resumable-checkpoint boundary (a multiple of 2*SAVE_INTERVAL optimizer
    # steps) -- that's what ends the phase-1 train.py call (the dataset
    # simply runs out), and it means at most one save_interval's worth of
    # steps needs re-doing if a session dies exactly at the boundary.
    # --max_steps is passed as the same final total (MAX_STEPS) on every
    # launch across both phases -- the LR / mean-recurrence schedules are
    # built fresh from --max_steps each launch but their step counters are
    # restored from the checkpoint, so passing the true final total everywhere
    # is what keeps warmup/cooldown coherent across the phase switch instead
    # of cooling down twice.
    _boundary_unit = 2 * SAVE_INTERVAL
    _phase1_end_step = max(_boundary_unit, (MAX_STEPS // 2 // _boundary_unit) * _boundary_unit)
    _phase1_tokens = _phase1_end_step * 64 * (1024 + 1)  # batch_size=64, block_len=max_length+1, matching the train.py call below
    _phase2_tokens = TOKEN_BUDGET - _phase1_tokens
    PHASES = [
        dict(
            name="phase1-heal",
            end_step=_phase1_end_step,
            token_budget=_phase1_tokens,
            weights=dict(fineweb_edu_weight=1.0, dclm_weight=0.0, cosmopedia_v2_weight=0.0, math_reasoning_weight=0.0, code_reasoning_weight=0.0),
        ),
        dict(
            name="phase2-diverse",
            end_step=MAX_STEPS,
            token_budget=_phase2_tokens,
            weights=dict(fineweb_edu_weight=0.50, dclm_weight=0.34, cosmopedia_v2_weight=0.04, math_reasoning_weight=0.07, code_reasoning_weight=0.05),
        ),
    ]
else:
    raise ValueError(f"Unknown MODE={MODE!r}, expected 'smoke_test' or 'full_run'")

print(f"MODE={MODE}: run_name={RUN_NAME}, max_steps={MAX_STEPS:,}, token_budget={TOKEN_BUDGET:,}, save_interval={SAVE_INTERVAL}, shuffle_buffer_size={SHUFFLE_BUFFER_SIZE}, max_mean_rec={MAX_MEAN_REC}, use_muon={USE_MUON}")
for _phase in PHASES:
    print(f"  phase '{_phase['name']}': end_step={_phase['end_step']:,}, token_budget={_phase['token_budget']:,}, weights={_phase['weights']}")


In [ ]:
import os

REPO_DIR = "/content/retrofitting-recurrence"
if not os.path.exists(REPO_DIR):
    !git clone --depth=1 https://github.com/usr-wwelsh/retrofitting-recurrence.git {REPO_DIR}
    assert _exit_code == 0, f"git clone failed with exit code {_exit_code} -- check the output above"
%cd {REPO_DIR}
!git pull
assert _exit_code == 0, f"git pull failed with exit code {_exit_code} -- check the output above"
!pip install -q -r requirements.txt
assert _exit_code == 0, f"pip install failed with exit code {_exit_code} -- check the output above for which package broke"


In [ ]:
import os
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

# Everything below is local Colab disk (ephemeral -- wiped when the session ends).
# Persistence across sessions comes entirely from the Hub repos, not this filesystem.
DATA_PATH = f"/content/data/{RUN_NAME}"
OUT_PATH = "/content/huginn_smollm2"

CHECKPOINT_REPO = "usr-wwelsh/smollm2-recurrent-checkpoints"   # private HF dataset repo, one file per run_name -- holds only the latest resumable checkpoint for each
FINAL_MODEL_REPO = "usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4-trained"   # public HF model repo -- only written to in "full_run" mode

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(OUT_PATH, exist_ok=True)
print(f"Local data dir: {DATA_PATH}")
print(f"Local checkpoint dir: {OUT_PATH}/{RUN_NAME}")
print(f"Resume checkpoints from/to: {CHECKPOINT_REPO}/{RUN_NAME}")

In [ ]:
wandb_disabled = "true"
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY").strip()  # a pasted-in secret with a trailing newline/space corrupts the key and wandb rejects it with a 401 that looks identical to a genuinely bad key
    wandb_disabled = "false"
    print("WANDB_API_KEY secret found -- wandb logging enabled.")
except Exception as e:
    print(f"No usable WANDB_API_KEY secret ({e}) -- wandb logging disabled, training still runs fine.")

In [ ]:
import glob

_SOURCE_LABELS = {
    "fineweb_edu_weight": "FineWeb-Edu",
    "dclm_weight": "DCLM",
    "cosmopedia_v2_weight": "Cosmopedia-v2",
    "math_reasoning_weight": "IFM Math-Reasoning",
    "code_reasoning_weight": "IFM Code-Reasoning",
}

for phase in PHASES:
    phase_path = f"{DATA_PATH}/{phase['name']}"
    if glob.glob(f"{phase_path}/*.parquet"):
        print(f"Phase '{phase['name']}': found existing packed shards in {phase_path}, skipping the mix step.")
        continue

    mix_desc = "/".join(_SOURCE_LABELS[k] for k, w in phase["weights"].items() if w > 0)
    print(f"Phase '{phase['name']}': no packed data found -- streaming and packing ~{phase['token_budget']:,} tokens of the {mix_desc} mix.")
    print("The first 'Resolving data files' lines are just the Hub listing each source's parquet shards -- normal, not a hang.")
    print("Then it goes quiet again until it's actually packed a full shard or hit log_every docs -- give it a minute rather than interrupting; -u below makes that output show up as it happens instead of buffering.")
    print("Resumes cleanly if interrupted anyway (already-written shards are kept, just re-run this cell) -- interrupting only loses whatever wasn't flushed to a shard yet.")

    weight_args = " ".join(f"--{k}={v}" for k, v in phase["weights"].items())
    !python -u mix_smollm2_corpus.py --save_path="{phase_path}" --token_budget={phase['token_budget']} --rows_per_shard={ROWS_PER_SHARD} --log_every={LOG_EVERY} --shuffle_buffer_size={SHUFFLE_BUFFER_SIZE} {weight_args}
    assert _exit_code == 0, f"mix_smollm2_corpus.py failed for phase '{phase['name']}' with exit code {_exit_code} -- scroll up for the traceback, then re-run this cell (already-written shards are kept)."

    # belt-and-suspenders: don't let a silently-empty mix step reach training
    assert glob.glob(f"{phase_path}/*.parquet"), f"No .parquet shards found in {phase_path} after the mix step for phase '{phase['name']}' -- check the output above for what went wrong."


In [ ]:
from huggingface_hub import hf_hub_download, HfApi
from huggingface_hub.errors import EntryNotFoundError

HUB_RESUME_DIR = None
api = HfApi()
if api.repo_exists(CHECKPOINT_REPO, repo_type="dataset"):
    try:
        resume_dir = f"{OUT_PATH}/resumed_checkpoint"
        hf_hub_download(repo_id=CHECKPOINT_REPO, repo_type="dataset", filename=f"{RUN_NAME}/chkpt.pt", local_dir=resume_dir)
        HUB_RESUME_DIR = f"{resume_dir}/{RUN_NAME}"
        print(f"Found a checkpoint for {RUN_NAME} on {CHECKPOINT_REPO} -- will resume from it.")
    except EntryNotFoundError:
        print(f"No checkpoint for {RUN_NAME} on {CHECKPOINT_REPO} yet -- starting a fresh run.")
else:
    print(f"No checkpoint on {CHECKPOINT_REPO} yet -- starting a fresh run.")

In [ ]:
import glob
import re

import torch


def latest_local_checkpoint():
    dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/checkpoint_*")
    if not dirs:
        return None, 0
    steps = {int(re.search(r"checkpoint_(\d+)", d).group(1)): d for d in dirs}
    best = max(steps)
    return steps[best], best


resume_path, resumed_step = latest_local_checkpoint()
if resume_path is None and HUB_RESUME_DIR is not None:
    resume_path = HUB_RESUME_DIR
    ckpt = torch.load(f"{resume_path}/chkpt.pt", map_location="cpu", weights_only=False)
    resumed_step = ckpt["agg_vars_dict"]["optimizer_step"]
    del ckpt

print(f"Resuming from step {resumed_step:,} (checkpoint: {resume_path})" if resume_path else "Starting fresh -- no checkpoint found.")

for phase in PHASES:
    if resumed_step >= phase["end_step"]:
        print(f"Phase '{phase['name']}' already complete (resumed at step {resumed_step:,} >= {phase['end_step']:,}) -- skipping.")
        continue

    print(f"Training phase '{phase['name']}' toward step {phase['end_step']:,} (of {MAX_STEPS:,} total)...")
    resume_flag = f"--resume_path={resume_path}" if resume_path else ""
    !python -u train.py \
        --run_name={RUN_NAME} \
        --out_path={OUT_PATH} \
        --model_name=usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4 \
        --hub_checkpoint_repo={CHECKPOINT_REPO} \
        --preprocessed_data_path={DATA_PATH}/{phase['name']} \
        --is_parquet_dataset=true \
        --max_length=1024 \
        --micro_batch_size={MICRO_BATCH_SIZE} \
        --batch_size=64 \
        --optim_config.lr=5e-5 \
        --scheduler_args.warmup=0.02 \
        --scheduler_args.cooldown=0.9 \
        --max_grad_norm=1.0 \
        --no_amp={no_amp} \
        --max_steps={MAX_STEPS} \
        --compile=false \
        --save_interval={SAVE_INTERVAL} \
        --wandb_disabled={wandb_disabled} \
        --wandb_project=smollm2-recurrent \
        --mean_recurrence_schedule.turn_on=true \
        --mean_recurrence_schedule.warmup=0.25 \
        --mean_recurrence_schedule.max_mean_rec={MAX_MEAN_REC} \
        --mean_backprop_depth_schedule.turn_on=true \
        --mean_backprop_depth_schedule.warmup=0.25 \
        --mean_backprop_depth_schedule.start=1 \
        --mean_backprop_depth_schedule.max_backprop={MAX_BACKPROP} \
        --muon.use_muon={str(USE_MUON).lower()} \
        {resume_flag}
    assert _exit_code == 0, f"train.py exited with code {_exit_code} during phase '{phase['name']}' -- check the traceback above. Cells below did not see a finished run."

    resume_path, resumed_step = latest_local_checkpoint()
    print(f"Phase '{phase['name']}' training call ended at step {resumed_step:,}.")


## Push final model to the Hub

Only runs in `full_run` mode, and only once training has actually reached `MAX_STEPS` in some session (checked via the checkpoint's saved step count, not just file presence) -- if the training cell above got cut off by a session timeout, re-run this notebook (Run All) to resume training first.

In [ ]:
import re

if MODE != "full_run":
    print(f"MODE={MODE!r} -- skipping the Hub push (only full_run publishes to {FINAL_MODEL_REPO}).")
else:
    ckpt_dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_*")
    if not ckpt_dirs:
        print("No eval checkpoint found yet -- nothing to push.")
    else:
        last_step = max(int(re.search(r"model_only_chkpt_(\d+)", d).group(1)) for d in ckpt_dirs)
        if last_step < MAX_STEPS:
            print(f"Latest checkpoint is at step {last_step:,}/{MAX_STEPS:,} -- training isn't finished yet, re-run this notebook to resume before pushing.")
        else:
            ckpt_path = f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_{last_step}"
            print(f"Training complete at step {last_step:,} -- pushing {ckpt_path} to {FINAL_MODEL_REPO}")
            api.create_repo(FINAL_MODEL_REPO, private=False, exist_ok=True)
            api.upload_folder(repo_id=FINAL_MODEL_REPO, folder_path=ckpt_path, commit_message=f"trained checkpoint @ step {last_step}")
            print("Done.")

## Eval

Sweeps recurrence depth on the latest local eval checkpoint and compares against base SmolLM2-360M, so you can see whether training is recovering toward (not stuck below) the base model. Works on any checkpoint reached so far -- doesn't require training to have finished, and works in either `MODE` (though `smoke_test` numbers are only a pipeline sanity check, not a real signal).

In [ ]:
import re

ckpt_dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_*")
if not ckpt_dirs:
    print("No model_only checkpoint found yet (training hasn't reached a save_interval boundary) -- nothing to eval.")
else:
    last_step = max(int(re.search(r"model_only_chkpt_(\d+)", d).group(1)) for d in ckpt_dirs)
    ckpt_path = f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_{last_step}"
    print(f"Evaluating {ckpt_path}")

    TASKS = "arc_easy,arc_challenge,hellaswag,mmlu,piqa,winogrande,gsm8k"  # gsm8k added alongside the math/code curriculum -- it's the cheapest lm-eval-harness signal for whether reasoning is actually improving, not just general-knowledge tasks. A full code-eval pass (humaneval etc.) needs sandboxed execution and belongs in a separate, deliberate run, not this quick per-checkpoint sanity check.
    for mean_recurrence in [1, 2, 4, 8]:
        out_dir = f"eval_outputs/{RUN_NAME}/step_{last_step}/mean_recurrence_{mean_recurrence}"
        !lm_eval --model hf \
            --model_args pretrained={ckpt_path},mean_recurrence={mean_recurrence},add_bos_token=True,dtype="float32",trust_remote_code=True \
            --tasks {TASKS} \
            --device cuda \
            --output_path {out_dir} \
            --batch_size auto

    !lm_eval --model hf \
        --model_args pretrained=HuggingFaceTB/SmolLM2-360M,add_bos_token=True,dtype="float32" \
        --tasks {TASKS} \
        --device cuda \
        --output_path eval_outputs/SmolLM2-360M-base \
        --batch_size auto